# RL Attack Path Simulation -- Colab (Train + Re-evaluate)
## MMAI 845 | Syed Ali Turab

**Instructions:**
1. Open this notebook in Google Colab
2. Go to Runtime > Change runtime type > Select **T4 GPU**
3. Run setup cells first
4. Choose one mode:
   - **Fast path (recommended):** run lint/tests + seeded evaluation on existing models (no retraining)
   - **Full path:** retrain baseline + stealth models, then evaluate
5. Download the `results/` folder when done

Training 500k steps per agent typically takes ~10-20 minutes each on a T4 GPU.

**Key features:**
- MaskablePPO (sb3-contrib) with NASim action masking
- DQN with manual Q-value masking for invalid actions
- Dense reward shaping for exploration
- Fully observable environment
- Seeded evaluation for reproducibility (`--seed 42`)

---
## 1. Setup

In [ ]:
# Check GPU availability
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Clone the repository
!git clone https://github.com/turaab97/rl-attack-path-simulation.git
%cd rl-attack-path-simulation

In [ ]:
# Install dependencies (includes sb3-contrib for MaskablePPO)
!pip install -e ".[dev]" -q

In [ ]:
# Verify environment and action masking
from environments.network_config import make_env, AI_INFRA_HOSTS
from agents.wrappers import IntActionWrapper, ActionMaskWrapper, DenseRewardWrapper
import numpy as np

env = make_env()
obs, info = env.reset()
print(f'Environment OK: obs={obs.shape}, actions={env.action_space.n}')
print(f'AI Infrastructure targets: {AI_INFRA_HOSTS}')
print(f'Fully observable: obs has {np.count_nonzero(obs)}/{obs.shape[0]} non-zero features')

# Verify action mask wrapper
masked_env = ActionMaskWrapper(IntActionWrapper(env))
masked_env.reset()
mask = masked_env.action_masks()
n_valid = int(mask.sum())
print(f'Action masking: {n_valid}/{env.action_space.n} valid actions from initial state')

# Verify dense reward wrapper
wrapped = DenseRewardWrapper(ActionMaskWrapper(IntActionWrapper(make_env())))
obs2, _ = wrapped.reset()
_, r1, _, _, _ = wrapped.step(0)
print(f'Dense reward wrapper OK (noop reward: {r1:.2f})')
wrapped.close()
masked_env.close()
print('All checks passed.')

---
## 2. Fast Path: Quality Gate (No Retraining)

Run this first to validate formatting, linting, and tests before evaluation/training.

In [ ]:
# Quick command flow gate
!black --check . && isort --check-only . && flake8 . && pytest tests/ -v --tb=short

---
## 3. Optional Full Retraining: Baseline (PPO + DQN, 500k steps)

Skip this if you already have trained models and only need re-evaluation.

In [ ]:
!python -m training.train --compare --timesteps 500000 --eval_freq 25000 --n_eval_episodes 10 --seed 42

---
## 4. Optional Full Retraining: Stealth (PPO + DQN, 500k steps)

In [ ]:
!python -m training.train --compare --stealth --timesteps 500000 \
    --detection_threshold 0.8 --detection_cost 0.1 --caught_penalty -100.0 --alpha 1.0 \
    --eval_freq 25000 --n_eval_episodes 10 --seed 42

In [ ]:
# Placeholder (safe no-op). Evaluation section starts below.

---
## 5. Evaluation (Seeded, Reproducible)

Run this cell to evaluate existing or newly trained models with a fixed seed.

In [ ]:
# Evaluate baseline models (seeded)
!python -m training.evaluate \
    --ppo_model results/ppo_baseline/final_model \
    --dqn_model results/dqn_baseline/final_model \
    --episodes 100 --seed 42

# Evaluate stealth models (seeded)
!python -m training.evaluate \
    --ppo_model results/ppo_stealth/final_model \
    --dqn_model results/dqn_stealth/final_model \
    --stealth --episodes 100 --seed 42

---
## 6. Generate Figures and Pentest Report

In [ ]:
!python analysis/generate_all_figures.py
!python -m analysis.report_generator --results_dir results/ --output results/pentest_report.md

---
## 7. Verify Output + Acceptance Checks

In [ ]:
import json
from pathlib import Path

results_dir = Path('results')
base_path = results_dir / 'eval_baseline.json'
stealth_path = results_dir / 'eval_stealth.json'

assert base_path.exists(), 'Missing results/eval_baseline.json'
assert stealth_path.exists(), 'Missing results/eval_stealth.json'

with open(base_path) as f:
    baseline = json.load(f)
with open(stealth_path) as f:
    stealth = json.load(f)

print('=== Acceptance Checks ===')

ppo_base_mean = baseline['ppo']['mean_reward']
ppo_base_std = baseline['ppo']['std_reward']
ok_ppo = abs(ppo_base_mean - (-100.0)) <= 20.0 and ppo_base_std <= 5.0
print(f"PPO baseline mean={ppo_base_mean:.2f}, std={ppo_base_std:.2f} -> {'PASS' if ok_ppo else 'CHECK'}")

ppo_stealth_mean = stealth['ppo']['mean_reward']
ppo_stealth_steps = stealth['ppo']['mean_steps']
ppo_stealth_catch = stealth['ppo']['catch_rate']
ok_stealth = abs(ppo_stealth_mean - (-109.9)) <= 5.0 and 8.0 <= ppo_stealth_steps <= 10.0 and ppo_stealth_catch >= 0.95
print(f"Stealth PPO mean={ppo_stealth_mean:.2f}, steps={ppo_stealth_steps:.1f}, catch={ppo_stealth_catch:.2f} -> {'PASS' if ok_stealth else 'CHECK'}")

print(f"DQN baseline std={baseline['dqn']['std_reward']:.2f} (higher variance vs PPO is expected in sparse masked setting)")

print('\n=== Baseline Summary ===')
for agent in ['ppo', 'dqn']:
    m = baseline[agent]
    print(f"{agent.upper()}: mean_reward={m['mean_reward']:.2f}, std={m['std_reward']:.2f}, mean_steps={m['mean_steps']:.1f}")

print('\n=== Stealth Summary ===')
for agent in ['ppo', 'dqn']:
    m = stealth[agent]
    print(f"{agent.upper()}: mean_reward={m['mean_reward']:.2f}, catch_rate={m['catch_rate']:.2f}, mean_steps={m['mean_steps']:.1f}")

print('\n=== Key Artifacts ===')
for rel in [
    'eval_baseline.json',
    'eval_stealth.json',
    'pentest_report.md',
    'plots/system_architecture.png',
    'plots/network_topology.png',
    'plots/attack_path_flow.png',
]:
    p = results_dir / rel
    print(f"{rel}: {'OK' if p.exists() else 'MISSING'}")

---
## 8. Download Results

Download the full `results/` directory to your local machine,
then copy it into your local repo clone to use with the analysis notebook.

In [ ]:
# Zip results for download
!zip -r /content/rl_results.zip results/

from google.colab import files
files.download('/content/rl_results.zip')
print('\nDownload started. Copy the results/ folder into your local repo clone.')

---

*Training notebook by Syed Ali Turab -- MMAI 845, Queen's University*